# 02. Entrenamiento de Modelos PINN y Comparación con Referencia FEM (FEniCSx)

Este cuaderno muestra cómo:
1. Listar las arquitecturas registradas en `ModelRegistry` con sus autores originales y modificaciones.
2. Entrenar un modelo clásico (`M1`), un enriquecido con cúspide escalada (`CUSPR`) y uno de restricciones exactas (`M1_EXACT`).
3. Ejecutar opcionalmente la referencia FEniCSx desde WSL2/nativo si está disponible.
4. Evaluar métricas y guardar artefactos compatibles en `frontend/workspace/runs/`.

In [ ]:
import sys
from pathlib import Path

# Localiza PINN_V4 automáticamente aunque se ejecute desde dentro de notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Raíz PINN_V4:", ROOT)

In [ ]:
import pinn_thermal.models  # registra todas las arquitecturas
from pinn_thermal.core.base_model import ModelRegistry
from pinn_thermal.core.evaluation import evaluate_model
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter
from pinn_thermal.visualization import GridSampler, StaticPlotter
from frontend.state import WorkspaceStore

for meta in ModelRegistry.describe_all():
    print(f"{meta['name']:15s} | {meta['family']:18s} | {meta['authors']} ({meta['year']}) [{meta['doi']}]")

In [ ]:
store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
problem = store.load_problem("mms_layered_y")

cfg = TrainConfig(
    adam_steps=250,
    lbfgs_steps=25,
    width=24,
    depth=3,
    device="auto",
    dtype="float32",
)
art_m1 = PINNTrainer(cfg).train(problem, "M1", store.runs_dir / "nb02_mms_layered_M1", overwrite=True)
art_cuspr = PINNTrainer(cfg).train(problem, "CUSPR", store.runs_dir / "nb02_mms_layered_CUSPR", overwrite=True)

for name, art in [("M1", art_m1), ("CUSPR", art_cuspr)]:
    ev = art.result["evaluation"]
    print(f"{name}: NRMSE={ev['nrmse_pct']:.3f}%, RMSE={ev['rmse_K']:.4f} K, balance={ev['balance']['global_derived_pct']:.3f}%")

# Ejemplo opcional FEniCSx (si WSL/dolfinx está disponible en la máquina)
fem_info = FEniCSxAdapter().is_available()
print("Estado FEniCSx:", fem_info)
if fem_info.get("available"):
    fem_art = FEniCSxAdapter().solve(problem, output_dir=store.runs_dir / "nb02_mms_layered_FEM_L1", level=1)
    print("FEniCSx Tmax:", fem_art.result["Tmax_C"], "°C | Balance:", fem_art.result["balance_pct"], "%")

In [ ]:
grid = GridSampler.sample_artifact_grid(art_cuspr, nx=80, ny=80)
out_png = store.runs_dir / "nb02_mms_layered_CUSPR" / "mapa_2d.png"
StaticPlotter.plot_temperature_field(grid, problem, title="nb02 CUSPR sobre mms_layered_y", output_path=out_png)
print("Lámina estática guardada en:", out_png)